<a href="https://colab.research.google.com/github/zamaraiomar-lang/Updated-Build/blob/main/TKS_BUILD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip -q install groq

In [31]:
import os, re, time, json, string, urllib.parse, urllib.request
from groq import Groq

try:
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    from getpass import getpass
    API_KEY = getpass("Paste your Groq API key: ")

MODEL = "openai/gpt-oss-20b"   # if you get a "model not found" error, use a model name from console.groq.com (Docs, then Models)
client = Groq(api_key=API_KEY, max_retries=0)

def failed_text(error):
    """When Groq refuses a reply it includes what the AI tried to write. Dig that text out of the error."""
    body = getattr(error, "body", None)
    if isinstance(body, dict):
        inner = body.get("error", body)
        if isinstance(inner, dict) and inner.get("failed_generation"):
            return str(inner["failed_generation"])
    found = re.search(r"failed_generation'?\"?: ?['\"](.*)['\"]\}\}", str(error), re.S)
    return found.group(1) if found else ""

def action_from_tool_call(text):
    """This AI sometimes tries to call a built-in tool, like {"name": "search", "arguments": {...}}.
    Turn that into the text action our loop understands, like Search[Danny Green]."""
    try:
        call = json.loads(text)
        name = str(call.get("name", "")).lower()
        args = call.get("arguments", {})
        if isinstance(args, str):
            args = json.loads(args)
    except Exception:
        return None
    value = ""
    for key in ("entity", "query", "keyword", "answer", "q", "text"):
        if args.get(key):
            value = str(args[key])
            break
    kind = "Lookup" if ("lookup" in name or "find" in name) else "Finish" if "finish" in name else "Search"
    return f"{kind}[{value}]"

def ask(prompt, stop=("\n",)):
    """Send text to the AI, get text back. Waits and retries if the free limit is hit."""
    messages = [
        {"role": "system", "content": "Continue the text you are given in exactly the same format. Reply with only the continuation. Never use tools or function calls: write actions as plain text like Search[name]."},
        {"role": "user", "content": prompt},
    ]
    extra = {"reasoning_effort": "low"} if "gpt-oss" in MODEL else {}
    for attempt in range(10):
        try:
            reply = client.chat.completions.create(model=MODEL, messages=messages, temperature=0,
                                                   max_completion_tokens=1024, stop=list(stop), **extra)
            text = reply.choices[0].message.content or ""
        except Exception as e:
            lowered = str(e).lower()
            if "failed_generation" in lowered:
                # Groq refused the reply because the AI tried to call a tool or wrote its thinking instead of plain text.
                bad = failed_text(e)
                action = action_from_tool_call(bad)
                step = re.search(r"Thought (\d+):$", prompt)
                if action and step:
                    return f" Let me look this up.\nAction {step.group(1)}: {action}"
                if action:
                    return " " + action
                text = bad                      # otherwise keep whatever text it wrote; the loop will treat it as a bad step
            elif "per day" in lowered or "tpd" in lowered:
                raise RuntimeError("You used up today's free Groq limit. Try again tomorrow (it resets daily).")
            elif any(w in lowered for w in ("429", "rate limit", "quota", "503", "unavailable", "overloaded")):
                wait = 20 * (attempt + 1)                      # default wait if Groq does not say how long
                try:
                    wait = float(e.response.headers.get("retry-after")) + 1   # Groq can tell us exactly how long to wait
                except Exception:
                    pass
                print(f"  (hit the free per-minute limit, waiting {wait:.0f} seconds. This is normal, do not stop it)", flush=True)
                time.sleep(wait)
                continue
            else:
                raise
        for s in stop:                          # cut the reply at the stop text, in case the AI ran past it
            text = text.split(s)[0]
        return text
    raise RuntimeError("Too many errors from the API")

print(ask("Say hello in five words.", stop=("\n\n",)))

Hello, how are you today?


In [34]:
def wiki_api(params):
    """Ask Wikipedia's public API a question and get the answer back as data."""
    url = "https://en.wikipedia.org/w/api.php?" + urllib.parse.urlencode({**params, "format": "json"})
    req = urllib.request.Request(url, headers={"User-Agent": "tks-react-build/0.2 (student project)"})
    return json.load(urllib.request.urlopen(req, timeout=15))

def sentences(text):
    """Cut a page into sentences."""
    out = []
    for paragraph in (p.strip() for p in text.split("\n") if p.strip()):
        out += paragraph.split(". ")
    return [s.strip().rstrip(".") + "." for s in out if s.strip()]

page_text = ""                                              # the page the agent is reading right now
lookup_state = {"keyword": None, "matches": [], "n": 0}     # where Lookup is up to

def search(entity):
    global page_text
    data = wiki_api({"action": "query", "titles": entity, "redirects": 1, "prop": "extracts", "explaintext": 1})
    for p in data.get("query", {}).get("pages", {}).values():
        text = p.get("extract") or ""
        if "missing" not in p and text and "may refer to" not in text[:400]:
            page_text = text
            lookup_state.update(keyword=None, matches=[], n=0)
            return " ".join(sentences(text)[:5])
    data = wiki_api({"action": "query", "list": "search", "srsearch": entity, "srlimit": 5})
    titles = [hit["title"] for hit in data.get("query", {}).get("search", [])]
    return f"Could not find {entity}. Similar: {titles}."

def lookup(keyword):
    if not page_text:
        return "No page yet. Use Search first."
    if lookup_state["keyword"] != keyword:
        lookup_state.update(keyword=keyword, matches=[s for s in sentences(page_text) if keyword.lower() in s.lower()], n=0)
    if lookup_state["n"] >= len(lookup_state["matches"]):
        return "No more results."
    lookup_state["n"] += 1
    return f"(Result {lookup_state['n']} / {len(lookup_state['matches'])}) " + lookup_state["matches"][lookup_state["n"] - 1]

def run_action(action):
    """Take text like 'Search[Colorado orogeny]' and do it. Returns (what we saw, final answer or None)."""
    m = re.fullmatch(r"(search|lookup|finish)\[(.*)\]", action.strip(), flags=re.I | re.S)
    if not m:
        return f"Invalid action: {action}", None
    kind, argument = m.group(1).lower(), m.group(2)
    if kind == "search":
        return search(argument), None
    if kind == "lookup":
        return lookup(argument), None
    return "Done.", argument

In [35]:
print(search("Colorado orogeny"))
print()
print(lookup("eastern sector"))
print()
print(search("High Plains"))        # an unclear name: look at what comes back

The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas. This took place from 1780 to 1650 million years ago (Mya), during the Paleoproterozoic (Statherian Period). It is recorded in the Colorado orogen, a >500-km-wide belt of oceanic arc rock that extends southward into New Mexico. The Colorado orogeny was likely part of the larger Yavapai orogeny. == Description ==.

(Result 1 / 1) The eastern sector extends into the High Plains and is called the Central Plains orogeny.

High Plains refers to either of two distinct land regions:. High Plains (United States), land region of the western Great Plains. High Plains (Australia), land region adjacent to the Great Dividing Range. == See also ==. Altiplano (disambiguation), Spanish for high plains.


In [45]:
INSTRUCTION = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.
"""

# These 6 examples come from the authors' repo (prompts/prompts_naive.json, key webthink_simple6).
EXAMPLES = """
Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Thought 1: I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.
Action 1: Search[Colorado orogeny]
Observation 1: The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.
Thought 2: It does not mention the eastern sector. So I need to look up eastern sector.
Action 2: Lookup[eastern sector]
Observation 2: (Result 1 / 1) The eastern sector extends into the High Plains and is called the Central Plains orogeny.
Thought 3: The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.
Action 3: Search[High Plains]
Observation 3: High Plains refers to one of two distinct land regions:
Thought 4: I need to instead search High Plains (United States).
Action 4: Search[High Plains (United States)]
Observation 4: The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130 m).[3]
Thought 5: High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.
Action 5: Finish[1,800 to 7,000 ft]
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Thought 1: The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.
Action 1: Search[Milhouse]
Observation 1: Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.
Thought 2: The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".
Action 2: Lookup[named after]
Observation 2: (Result 1 / 1) Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous.
Thought 3: Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.
Action 3: Finish[Richard Nixon]
Question: Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture?
Thought 1: I need to search Adam Clayton Powell and The Saimaa Gesture, and find which documentary is about Finnish rock groups.
Action 1: Search[Adam Clayton Powell]
Observation 1: Could not find [Adam Clayton Powell]. Similar: ['Adam Clayton Powell III', 'Seventh Avenue (Manhattan)', 'Adam Clayton Powell Jr. State Office Building', 'Isabel Washington Powell', 'Adam Powell', 'Adam Clayton Powell (film)', 'Giancarlo Esposito'].
Thought 2: To find the documentary, I can search Adam Clayton Powell (film).
Action 2: Search[Adam Clayton Powell (film)]
Observation 2: Adam Clayton Powell is a 1989 American documentary film directed by Richard Kilberg.
The film is about the rise and fall of influential African-American politician Adam Clayton Powell Jr.[3][4] It was later aired as part of the PBS series The American Experience.
Thought 3: Adam Clayton Powell (film) is a documentary about an African-American politician, not Finnish rock groups. So the documentary about Finnish rock groups must instead be The Saimaa Gesture.
Action 3: Finish[The Saimaa Gesture]
Question: What profession does Nicholas Ray and Elia Kazan have in common?
Thought 1: I need to search Nicholas Ray and Elia Kazan, find their professions, then find the profession they have in common.
Action 1: Search[Nicholas Ray]
Observation 1: Nicholas Ray (born Raymond Nicholas Kienzle Jr., August 7, 1911 – June 16, 1979) was an American film director, screenwriter, and actor best known for the 1955 film Rebel Without a Cause.
Thought 2: Professions of Nicholas Ray are director, screenwriter, and actor. I need to search Elia Kazan next and find his professions.
Action 2: Search[Elia Kazan]
Observation 2: Elia Kazan was an American film and theatre director, producer, screenwriter and actor.
Thought 3: Professions of Elia Kazan are director, producer, screenwriter, and actor. So profession Nicholas Ray and Elia Kazan have in common is director, screenwriter, and actor.
Action 3: Finish[director, screenwriter, actor]
Question: Which magazine was started first Arthur's Magazine or First for Women?
Thought 1: I need to search Arthur's Magazine and First for Women, and find which was started first.
Action 1: Search[Arthur's Magazine]
Observation 1: Arthur's Magazine (1844-1846) was an American literary periodical published in Philadelphia in the 19th century.
Thought 2: Arthur's Magazine was started in 1844. I need to search First for Women next.
Action 2: Search[First for Women]
Observation 2: First for Women is a woman's magazine published by Bauer Media Group in the USA.[1] The magazine was started in 1989.
Thought 3: First for Women was started in 1989. 1844 (Arthur's Magazine) < 1989 (First for Women), so Arthur's Magazine was started first.
Action 3: Finish[Arthur's Magazine]
Question: Were Pavel Urysohn and Leonid Levin known for the same type of work?
Thought 1: I need to search Pavel Urysohn and Leonid Levin, find their types of work, then find if they are the same.
Action 1: Search[Pavel Urysohn]
Observation 1: Pavel Samuilovich Urysohn (February 3, 1898 – August 17, 1924) was a Soviet mathematician who is best known for his contributions in dimension theory.
Thought 2: Pavel Urysohn is a mathematician. I need to search Leonid Levin next and find its type of work.
Action 2: Search[Leonid Levin]
Observation 2: Leonid Anatolievich Levin is a Soviet-American mathematician and computer scientist.
Thought 3: Leonid Levin is a mathematician and computer scientist. So Pavel Urysohn and Leonid Levin have the same type of work.
Action 3: Finish[yes]
"""

In [36]:
def clean_reply(reply, i):
    """Keep only the first Thought and Action the AI wrote. The AI sometimes rambles, writes its own
    Observations, or repeats step labels. Anything after 'Observation' is invented, so we cut it off."""
    reply = re.split(r"\n\s*Observation", reply)[0]
    reply = re.sub(r"^\s*Thought\s*\d*\s*:\s*", "", reply)
    parts = re.split(r"\n\s*Action\s*\d*\s*:", reply, maxsplit=1)
    thought = parts[0].strip().split("\n")[0]
    action_text = parts[1] if len(parts) > 1 else reply
    found = re.search(r"(Search|Lookup|Finish)\[(.+?)\]", action_text, flags=re.I | re.S)
    if not found and len(parts) > 1:
        found = re.search(r"(Search|Lookup|Finish)\[(.+?)\]", reply, flags=re.I | re.S)
    action = f"{found.group(1).capitalize()}[{found.group(2).strip()}]" if found else None
    if thought.startswith("{"):
        thought = "I need to look this up."
    return thought[:300], action

def react(question, max_steps=7, show=True):
    """Run the ReAct loop on one question and return the agent's answer."""
    global page_text
    page_text = ""
    lookup_state.update(keyword=None, matches=[], n=0)
    prompt = INSTRUCTION + EXAMPLES + "Question: " + question + "\n"
    for i in range(1, max_steps + 1):
        # 1. ask the AI for a Thought and an Action
        reply = ask(prompt + f"Thought {i}:", stop=(f"\nObservation {i}:",))
        thought, action = clean_reply(reply, i)
        if action is None:
            # No usable action: ask for the action alone, once.
            reply2 = ask(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=("\n",))
            _, action = clean_reply(reply2, i)
        if action is None:
            action = "Invalid"
        # 2. do the action and see what happens
        if action == "Invalid":
            observation, answer = "Invalid action. Use Search[...], Lookup[...] or Finish[...].", None
        else:
            observation, answer = run_action(action)
        # 3. add this clean step to the prompt so the AI can see it next time
        step = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {observation}\n"
        prompt += step
        if show:
            print(step)
        if answer is not None:
            return answer
    return ""        # ran out of steps without an answer

In [37]:
print("ANSWER:", react("Who is older Danny Green or James Worthy?"))

Thought 1: I need to look this up.
Action 1: Search[Danny Green (basketball)]
Observation 1: Daniel Richard Green Jr. (born June 22, 1987) is an American former professional basketball player. In his NBA career, Green played for six teams. As of 2025, Green is one of just four players in history to have won NBA championships with three different teams; he won titles with the San Antonio Spurs in 2014, the Toronto Raptors in 2019, and the Los Angeles Lakers in 2020. Green spent his college basketball career at the University of North Carolina (UNC) from 2005 to 2009.

Thought 2: I need to look this up.
Action 2: Search[James Worthy]
Observation 2: James Ager Worthy (born February 27, 1961) is an American former professional basketball player. Nicknamed "Big Game James", he played his entire 12-year professional career with the Los Angeles Lakers in the National Basketball Association (NBA). Worthy was a seven-time NBA All-Star, a two-time All-NBA Team member who won three NBA championsh

In [38]:
# RESTORE CELL: questions + scoring + clean plain-AI function + your 7 saved results
QUESTIONS = [
 ["What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?", "Beyond the Clouds"],
 ["Who created the show with Wendy Schaal doing the voice of Francine?", "Seth MacFarlane"],
 ["What objects were carried into battle by these naval ships for qhich the QF 6-pounder Hotchkiss were introduced to defend against?", "torpedoes"],
 ["Who is older Danny Green or James Worthy?", "James Worthy"],
 ["Author David Chanoff has collaborated with a U.S. Navy admiral who served as the ambassador to the United Kingdom under which President?", "Bill Clinton"],
 ["What was the first year a scientific journal published by an organization located in the Bronx was published?", "1909"],
 ["Geoff LaTulippe is an American writer whose best-known work was directed by whom?", "Nanette Burstein"],
 ["Are Hoodoo Gurus and Pierre Bouvier of the same nationality?", "no"],
 ["Hardley Flood is an area of lagoons that support populations of the waterfowl that are a medium sized version of what animal?", "duck"],
 ["The Twelfth United States Army Group commander was the first chairman of what?", "Joint Chiefs of Staff"],
]

# A cleaner "plain AI" baseline: no search words in the instructions, so it should not try to use tools.
def plain_answer(question):
    messages = [
        {"role": "system", "content": "You have no tools and cannot search. Answer the question from memory in a few words. Reply with only the answer."},
        {"role": "user", "content": question},
    ]
    for attempt in range(6):
        try:
            reply = client.chat.completions.create(model=MODEL, messages=messages, temperature=0,
                                                   max_completion_tokens=1024, reasoning_effort="low")
            text = (reply.choices[0].message.content or "").strip()
        except Exception as e:
            lowered = str(e).lower()
            if "per day" in lowered or "tpd" in lowered:
                raise RuntimeError("Daily free Groq limit reached. " + str(e)[:200])
            if "failed_generation" in lowered:
                continue                                  # the AI tried to use a tool again: ask once more
            if any(w in lowered for w in ("429", "rate limit", "503", "unavailable", "overloaded")):
                wait = 20
                try:
                    wait = float(e.response.headers.get("retry-after")) + 1
                except Exception:
                    pass
                print(f"  (waiting {wait:.0f} seconds for the free limit)", flush=True)
                time.sleep(wait)
                continue
            raise
        if text and not re.match(r"^(search|lookup|finish|calculate)\[", text, flags=re.I):
            return text
    return ""        # no usable answer after several tries

import string
def normalize(s):
    s = s.lower()
    s = "".join(c for c in s if c not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

# Your 7 saved results, copied from the table you got earlier (no AI tokens used).
saved = [
    (1, "The Last Stand", ""),
    (2, "Matt Groening.", "Seth MacFarlane"),
    (3, "Torpedo-boats (and the torpedoes they carried).", "torpedo boats"),
    (4, "James Worthy.", "James Worthy"),
    (5, "George W. Bush", "George H. W. Bush"),
    (6, "1900", ""),
    (7, "Michael Mann", "Geoff LaTulippe"),
]
rows = []
for n, plain, agent in saved:
    question, gold = QUESTIONS[n - 1]
    rows.append({"#": n, "question": question[:70], "gold": gold, "plain AI": plain, "ReAct": agent,
                 "plain correct": normalize(plain) == normalize(gold), "ReAct correct": normalize(agent) == normalize(gold)})
print("Restored", len(rows), "results. Plain AI:", sum(r["plain correct"] for r in rows), "of 7. ReAct:", sum(r["ReAct correct"] for r in rows), "of 7.")

Restored 7 results. Plain AI: 1 of 7. ReAct: 2 of 7.


In [18]:
# Carry on the 10-question test from where it stopped, using the clean plain-AI baseline.
for n, (question, gold) in enumerate(QUESTIONS, 1):
    if n <= len(rows):
        continue                                   # already done
    plain = plain_answer(question)
    agent = react(question, show=False)
    rows.append({"#": n, "question": question[:70], "gold": gold, "plain AI": plain, "ReAct": agent,
                 "plain correct": normalize(plain) == normalize(gold), "ReAct correct": normalize(agent) == normalize(gold)})
    print(f"{n}/10 done")

import pandas as pd
results = pd.DataFrame(rows)
display(results)
print("Questions finished:", len(results))
print("Plain AI:    ", int(results["plain correct"].sum()), "of", len(results))
print("ReAct:       ", int(results["ReAct correct"].sum()), "of", len(results))

8/10 done
9/10 done
  (hit the free per-minute limit, waiting 15 seconds. This is normal, do not stop it)
  (hit the free per-minute limit, waiting 15 seconds. This is normal, do not stop it)
  (hit the free per-minute limit, waiting 15 seconds. This is normal, do not stop it)
10/10 done


,#,question,gold,plain AI,ReAct,plain correct,ReAct correct
0,1,What movie did actress Irene Jacob complete be...,Beyond the Clouds,The Last Stand,,False,False
1,2,Who created the show with Wendy Schaal doing t...,Seth MacFarlane,Matt Groening.,Seth MacFarlane,False,True
2,3,What objects were carried into battle by these...,torpedoes,Torpedo-boats (and the torpedoes they carried).,torpedo boats,False,False
3,4,Who is older Danny Green or James Worthy?,James Worthy,James Worthy.,James Worthy,True,True
4,5,Author David Chanoff has collaborated with a U...,Bill Clinton,George W. Bush,George H. W. Bush,False,False
5,6,What was the first year a scientific journal p...,1909,1900,,False,False
6,7,Geoff LaTulippe is an American writer whose be...,Nanette Burstein,Michael Mann,Geoff LaTulippe,False,False
7,8,Are Hoodoo Gurus and Pierre Bouvier of the sam...,no,No.,No,True,True
8,9,Hardley Flood is an area of lagoons that suppo...,duck,mallard,geese,False,False
9,10,The Twelfth United States Army Group commander...,Joint Chiefs of Staff,Joint Chiefs of Staff,Joint Chiefs of Staff,True,True


Questions finished: 10
Plain AI:     3 of 10
ReAct:        4 of 10


In [42]:
import ast, operator

# ---- New tool: Calculate[expression] (this is my own addition, not from the paper) ----
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def calculate(expression):
    """Do simple maths safely. Only numbers and + - * / ** are allowed."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported")
    try:
        result = ev(ast.parse(expression.replace(",", "").strip(), mode="eval").body)
        return str(round(result, 6)).rstrip("0").rstrip(".") if isinstance(result, float) else str(result)
    except Exception:
        return "Error: could not calculate that. Use numbers and + - * / ** only."

# Tell the AI the new tool exists (only once, so running this cell twice does no harm)
if "Calculate[" not in INSTRUCTION:
    INSTRUCTION = INSTRUCTION.replace("Action can be three types:", "Action can be four types:").replace(
        "Here are some examples.",
        "(4) Calculate[expression], which does simple maths such as Calculate[1969 - 1889] and returns the result.\nHere are some examples.")
if "Calculate[" not in EXAMPLES:
    EXAMPLES = EXAMPLES + """
Question: How many years after the Eiffel Tower was completed did Apollo 11 land on the Moon?
Thought 1: I need to find when the Eiffel Tower was completed, then when Apollo 11 landed, then subtract.
Action 1: Search[Eiffel Tower]
Observation 1: The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.
Thought 2: The Eiffel Tower was completed in 1889. Next I need the year Apollo 11 landed.
Action 2: Search[Apollo 11]
Observation 2: Apollo 11 was the first spaceflight to land humans on the Moon. Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed on July 20, 1969.
Thought 3: Apollo 11 landed in 1969. I can subtract 1889 from 1969 with the calculator.
Action 3: Calculate[1969 - 1889]
Observation 3: 80
Thought 4: The answer is 80 years.
Action 4: Finish[80]
"""

def run_action(action):
    """Take text like 'Search[Colorado orogeny]' and do it. Returns (what we saw, final answer or None)."""
    m = re.fullmatch(r"(search|lookup|calculate|finish)\[(.*)\]", action.strip(), flags=re.I | re.S)
    if not m:
        return f"Invalid action: {action}", None
    kind, argument = m.group(1).lower(), m.group(2)
    if kind == "search":
        return search(argument), None
    if kind == "lookup":
        return lookup(argument), None
    if kind == "calculate":
        return calculate(argument), None
    return "Done.", argument

def clean_reply(reply, i):
    """Keep only the first Thought and Action the AI wrote. Anything after 'Observation' is invented, so we cut it off."""
    reply = re.split(r"\n\s*Observation", reply)[0]
    reply = re.sub(r"^\s*Thought\s*\d*\s*:\s*", "", reply)
    parts = re.split(r"\n\s*Action\s*\d*\s*:", reply, maxsplit=1)
    thought = parts[0].strip().split("\n")[0]
    action_text = parts[1] if len(parts) > 1 else reply
    pattern = r"(Search|Lookup|Calculate|Finish)\[(.+?)\]"
    found = re.search(pattern, action_text, flags=re.I | re.S)
    if not found and len(parts) > 1:
        found = re.search(pattern, reply, flags=re.I | re.S)
    action = f"{found.group(1).capitalize()}[{found.group(2).strip()}]" if found else None
    if thought.startswith("{"):
        thought = "I need to look this up."
    return thought[:300], action

print("Calculator tool added. Test:", calculate("1969 - 1889"))

Calculator tool added. Test: 80


In [20]:
print("ANSWER:", react("How many years after the Statue of Liberty was dedicated was the Eiffel Tower completed?"))

Thought 1: I need the dedication date of the Statue of Liberty and the completion date of the Eiffel Tower, then subtract.
Action 1: Search[Statue of Liberty dedication]
Observation 1: Could not find Statue of Liberty dedication. Similar: ['Statue of Liberty', 'Eugène Spuller', 'Statue of Liberty commemorative coins', 'Statue of Liberty Museum', 'Replicas of the Statue of Liberty'].

Thought 2: I need to find the dedication date of the Statue of Liberty.
Action 2: Search[Statue of Liberty dedication]
Observation 2: Could not find Statue of Liberty dedication. Similar: ['Statue of Liberty', 'Eugène Spuller', 'Replicas of the Statue of Liberty', 'Statue of Liberty commemorative coins', 'Liberty Island'].

Thought 3: The Statue of Liberty was dedicated on October 28, 1886. The Eiffel Tower was completed in 1889.
Action 3: Calculate[1889 - 1886]
Observation 3: 3

  (hit the free per-minute limit, waiting 6 seconds. This is normal, do not stop it)
Thought 4: Action 4: Finish[3]
Action 4: Fi

In [41]:
import ast

# ---- Extension 2: two fixes to the agent (my own changes, not from the paper) ----
# Fix 1: if a Search says "Could not find ... Similar: [...]", automatically try the first suggested title.
# Fix 2: if the agent runs out of steps, make it give its best guess instead of a blank answer.

def retry_with_suggestion(action, observation):
    """If this was a failed Search, try the first 'Similar' title and add what we saw to the observation."""
    kind = action.split("[", 1)[0].lower()
    if kind != "search" or not observation.startswith("Could not find"):
        return observation
    found = re.search(r"Similar: (\[.*\])\.$", observation, flags=re.S)
    if not found:
        return observation
    try:
        titles = ast.literal_eval(found.group(1))
    except Exception:
        return observation
    entity = action[action.index("[") + 1:-1]
    if not titles or titles[0].lower() == entity.lower():
        return observation
    return observation + f" (Automatically trying {titles[0]} instead.) " + search(titles[0])

def clean_forced(text):
    """The AI sometimes repeats the label we started for it, like 'Action 8: Finish[answer'. Keep only the answer."""
    text = re.sub(r"^.*?Finish\[", "", text, flags=re.I | re.S)
    return text.strip().strip("]").strip()

def react_v2(question, max_steps=7, show=True):
    """The same loop as react(), plus the two fixes above."""
    global page_text
    page_text = ""
    lookup_state.update(keyword=None, matches=[], n=0)
    prompt = INSTRUCTION + EXAMPLES + "Question: " + question + "\n"
    for i in range(1, max_steps + 1):
        reply = ask(prompt + f"Thought {i}:", stop=(f"\nObservation {i}:",))
        thought, action = clean_reply(reply, i)
        if action is None:
            reply2 = ask(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=("\n",))
            _, action = clean_reply(reply2, i)
        if action is None:
            action = "Invalid"
        if action == "Invalid":
            observation, answer = "Invalid action. Use Search[...], Lookup[...] or Finish[...].", None
        else:
            observation, answer = run_action(action)
            observation = retry_with_suggestion(action, observation)          # fix 1
        step = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {observation}\n"
        prompt += step
        if show:
            print(step)
        if answer is not None:
            return answer
    # fix 2: out of steps, so ask for a best guess
    n = max_steps + 1
    forced = clean_forced(ask(prompt + f"Thought {n}: I have no steps left, so I must answer now with my best guess.\nAction {n}: Finish[",
                              stop=("]", "\n")))
    if show:
        print(f"Thought {n}: I have no steps left, so I must answer now with my best guess.\nAction {n}: Finish[{forced}]\n")
    return forced

print("Fixed agent ready: react_v2(...)")

Fixed agent ready: react_v2(...)


In [46]:
# Rerun the questions the first agent got wrong, with the fixed agent. Run this cell again later to carry on.
# IMPORTANT: this should use the paper's 3-tool prompt. If you ran the calculator cell, first re-run the examples cell
# (the one starting INSTRUCTION = """Solve a question answering task...) to switch the calculator prompt off.
if "Calculate[" in INSTRUCTION:
    raise RuntimeError("The calculator prompt is on. Re-run the examples cell (INSTRUCTION = ...) first, then run this cell again.")

TO_RERUN = [1, 3, 5, 6, 7, 9]       # the questions the first agent got wrong
BATCH = 3                            # how many to do each time you run this cell (saves tokens)

if "rows_v2" not in globals():
    rows_v2 = []
done = {r["#"] for r in rows_v2}
todo = [n for n in TO_RERUN if n not in done][:BATCH]
for n in todo:
    question, gold = QUESTIONS[n - 1]
    first = rows[n - 1]
    improved = react_v2(question, show=False)
    rows_v2.append({"#": n, "gold": gold, "first agent": first["ReAct"], "fixed agent": improved,
                    "first correct": first["ReAct correct"], "fixed correct": normalize(improved) == normalize(gold)})
    print(f"question {n} done")

import pandas as pd
table = pd.DataFrame(sorted(rows_v2, key=lambda r: r["#"]))
display(table)
print("Rerun so far:", len(table), "of", len(TO_RERUN))
print("First agent correct:", int(table["first correct"].sum()), "| Fixed agent correct:", int(table["fixed correct"].sum()))

question 1 done
question 3 done
  (hit the free per-minute limit, waiting 6 seconds. This is normal, do not stop it)
question 5 done


,#,gold,first agent,fixed agent,first correct,fixed correct
0,1,Beyond the Clouds,,The Last of the Mohicans,False,False
1,3,torpedoes,torpedo boats,torpedo boats,False,False
2,5,Bill Clinton,George H. W. Bush,George H. W. Bush,False,False


Rerun so far: 3 of 6
First agent correct: 0 | Fixed agent correct: 0


In [26]:
for r in rows_v2:
    r["fixed agent"] = clean_forced(r["fixed agent"])
    r["fixed correct"] = normalize(r["fixed agent"]) == normalize(r["gold"])
display(pd.DataFrame(sorted(rows_v2, key=lambda r: r["#"])))

,#,gold,first agent,fixed agent,first correct,fixed correct
0,1,Beyond the Clouds,,The Last of the Mohicans,False,False
1,3,torpedoes,torpedo boats,torpedo boats,False,False
2,5,Bill Clinton,George H. W. Bush,George H. W. Bush,False,False


In [22]:
import urllib.error

def wiki_api(params):
    """Ask Wikipedia's public API a question. If Wikipedia says 'too many requests', wait and try again."""
    url = "https://en.wikipedia.org/w/api.php?" + urllib.parse.urlencode({**params, "format": "json"})
    req = urllib.request.Request(url, headers={"User-Agent": "tks-react-build/0.2 (student project)"})
    for attempt in range(6):
        try:
            return json.load(urllib.request.urlopen(req, timeout=15))
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503, 504):
                wait = 5 * (attempt + 1)
                try:
                    wait = max(wait, int(e.headers.get("Retry-After")))
                except Exception:
                    pass
                print(f"  (Wikipedia asked me to slow down, waiting {wait} seconds)", flush=True)
                time.sleep(wait)
                continue
            raise
        except Exception:
            if attempt == 5:
                raise
            time.sleep(3)
    raise RuntimeError("Wikipedia kept refusing the request. Try again in a few minutes.")

print("Wikipedia calls now wait and retry.")

Wikipedia calls now wait and retry.


In [30]:
print("ANSWER:", react_v2(QUESTIONS[4][0], show=True))

Thought 1: I need to look this up.
Action 1: Search[David Chanoff]
Observation 1: David Chanoff (born November 15, 1943, in Philadelphia) is an American author of non-fiction work. His work has typically involved collaborations with the principal protagonist of the work concerned. His collaborators have included Augustus A. White, Joycelyn Elders, Đoàn Văn Toại, William J. Crowe, Ariel Sharon, Kenneth Good and Felix Zandman, among others.

Thought 2: Finish[George H. W. Bush]
Action 2: Finish[George H. W. Bush]
Observation 2: Done.

ANSWER: George H. W. Bush


In [23]:
# ---- Extension 3: don't let the agent finish until something it has read supports its answer ----
# (my own change, not from the paper). Needs the "Fixed agent" cell (retry_with_suggestion, clean_forced) to be run first.

def is_supported(answer, observations):
    """True if every word of the answer appears somewhere in what the agent has read so far."""
    a = normalize(answer)
    if not a or a in ("yes", "no"):
        return True                                   # yes/no answers can't be checked this way
    text = normalize(" ".join(observations))
    return all(word in text for word in a.split())

def react_v3(question, max_steps=7, show=True):
    """react_v2 plus: the first time the agent tries to finish with an unsupported answer, send it back to search."""
    global page_text
    page_text = ""
    lookup_state.update(keyword=None, matches=[], n=0)
    prompt = INSTRUCTION + EXAMPLES + "Question: " + question + "\n"
    seen, warned = [], False
    for i in range(1, max_steps + 1):
        reply = ask(prompt + f"Thought {i}:", stop=(f"\nObservation {i}:",))
        thought, action = clean_reply(reply, i)
        if action is None:
            reply2 = ask(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=("\n",))
            _, action = clean_reply(reply2, i)
        if action is None:
            action = "Invalid"
        if action == "Invalid":
            observation, answer = "Invalid action. Use Search[...], Lookup[...] or Finish[...].", None
        else:
            observation, answer = run_action(action)
            observation = retry_with_suggestion(action, observation)
            if answer is None:
                seen.append(observation)
            elif not warned and not is_supported(answer, seen):          # the new check
                warned = True
                observation = "Not finished: nothing you have read supports that answer yet. Search for the missing fact first."
                answer = None
        step = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {observation}\n"
        prompt += step
        if show:
            print(step)
        if answer is not None:
            return answer
    n = max_steps + 1
    forced = clean_forced(ask(prompt + f"Thought {n}: I have no steps left, so I must answer now with my best guess.\nAction {n}: Finish[",
                              stop=("]", "\n")))
    if show:
        print(f"Thought {n}: I have no steps left, so I must answer now with my best guess.\nAction {n}: Finish[{forced}]\n")
    return forced

print("Agent v3 ready: react_v3(...)")

Agent v3 ready: react_v3(...)


In [32]:
print("ANSWER:", react_v3(QUESTIONS[4][0], show=True))

Thought 1: I need to look this up.
Action 1: Search[David Chanoff]
Observation 1: David Chanoff (born November 15, 1943, in Philadelphia) is an American author of non-fiction work. His work has typically involved collaborations with the principal protagonist of the work concerned. His collaborators have included Augustus A. White, Joycelyn Elders, Đoàn Văn Toại, William J. Crowe, Ariel Sharon, Kenneth Good and Felix Zandman, among others.

Thought 2: Finish[George H. W. Bush]
Action 2: Finish[George H. W. Bush]
Observation 2: Not finished: nothing you have read supports that answer yet. Search for the missing fact first.

Thought 3: I need to look this up.
Action 3: Search[William J. Crowe ambassador to United Kingdom]
Observation 3: Could not find William J. Crowe ambassador to United Kingdom. Similar: ['William J. Crowe', 'List of ambassadors of the United States to the United Kingdom', 'List of diplomats of the United Kingdom to the Ottoman Empire', 'List of ambassadors of the Unite

In [50]:
# Rerun the questions the first agent got wrong with react_v3 (the version with the evidence check).
# Run this cell again later to carry on. Needs the "Fixed agent" cell and the react_v3 cell to be run first.
if "Calculate[" in INSTRUCTION:
    raise RuntimeError("The calculator prompt is on. Re-run the examples cell (INSTRUCTION = ...) first, then run this cell again.")

TO_RERUN = [1, 3, 5, 6, 7, 9]       # the questions the first agent got wrong
BATCH = 3                            # how many to do each time you run this cell (saves tokens)

if "rows_v3" not in globals():
    # Q5 is already done: it came from the test run where you watched the steps (answer: Bill Clinton)
    rows_v3 = [{"#": 5, "gold": QUESTIONS[4][1], "first agent": rows[4]["ReAct"], "v3 agent": "Bill Clinton",
                "first correct": rows[4]["ReAct correct"], "v3 correct": normalize("Bill Clinton") == normalize(QUESTIONS[4][1])}]
done = {r["#"] for r in rows_v3}
todo = [n for n in TO_RERUN if n not in done][:BATCH]
for n in todo:
    question, gold = QUESTIONS[n - 1]
    first = rows[n - 1]
    improved = react_v3(question, show=False)
    rows_v3.append({"#": n, "gold": gold, "first agent": first["ReAct"], "v3 agent": improved,
                    "first correct": first["ReAct correct"], "v3 correct": normalize(improved) == normalize(gold)})
    print(f"question {n} done")

import pandas as pd
table = pd.DataFrame(sorted(rows_v3, key=lambda r: r["#"]))
display(table)
print("Rerun so far:", len(table), "of", len(TO_RERUN))
print("First agent correct:", int(table["first correct"].sum()), "| v3 agent correct:", int(table["v3 correct"].sum()))

question 9 done


,#,gold,first agent,v3 agent,first correct,v3 correct
0,1,Beyond the Clouds,,The Last of the Mohicans,False,False
1,3,torpedoes,torpedo boats,torpedo boats,False,False
2,5,Bill Clinton,George H. W. Bush,Bill Clinton,False,True
3,6,1909,,1900,False,False
4,7,Nanette Burstein,Geoff LaTulippe,Nanette Burstein,False,True
5,9,duck,geese,duck,False,True


Rerun so far: 6 of 6
First agent correct: 0 | v3 agent correct: 3


In [34]:
print("ANSWER:", react_v3(QUESTIONS[5][0], show=True))

Thought 1: I need to look this up.
Action 1: Search[Bronx scientific journal]
Observation 1: Could not find Bronx scientific journal. Similar: ['Bronx High School of Science', 'Bronx Zoo', 'List of Bronx High School of Science alumni', 'New York Botanical Garden', 'Pelham Bay Park']. (Automatically trying Bronx High School of Science instead.) The Bronx High School of Science is a public specialized high school founded in 1938 in the Bronx in New York City. Admission involves passing the Specialized High Schools Admissions Test. The school is operated by the New York City Department of Education. Founded by science educator Morris Meister who served as its first principal, Bronx Science is located in what is now Kingsbridge Heights, also known as Jerome Park, a neighborhood in the northwest portion of the Bronx. Known for its focus on mathematics and science, Bronx Science also emphasizes the humanities and social sciences.

Thought 2: I need to look this up.
Action 2: Search[Bronx sci

In [24]:
# ---- Extension 4: stop the agent repeating the exact same action (my own change, not from the paper) ----
# Needs the "Fixed agent" cell and the react_v3 cell (is_supported) to be run first.

def react_v4(question, max_steps=7, show=True):
    """react_v3 plus: if the agent repeats an exact action it already did, tell it to try something different."""
    global page_text
    page_text = ""
    lookup_state.update(keyword=None, matches=[], n=0)
    prompt = INSTRUCTION + EXAMPLES + "Question: " + question + "\n"
    seen, warned, tried = [], False, set()
    for i in range(1, max_steps + 1):
        reply = ask(prompt + f"Thought {i}:", stop=(f"\nObservation {i}:",))
        thought, action = clean_reply(reply, i)
        if action is None:
            reply2 = ask(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=("\n",))
            _, action = clean_reply(reply2, i)
        if action is None:
            action = "Invalid"
        if action == "Invalid":
            observation, answer = "Invalid action. Use Search[...], Lookup[...] or Finish[...].", None
        elif action.lower() in tried and not action.lower().startswith("finish"):          # the new check
            observation, answer = "You already tried that exact action and the result will be the same. Try different words, or search a different page.", None
        else:
            tried.add(action.lower())
            observation, answer = run_action(action)
            observation = retry_with_suggestion(action, observation)
            if answer is None:
                seen.append(observation)
            elif not warned and not is_supported(answer, seen):
                warned = True
                observation = "Not finished: nothing you have read supports that answer yet. Search for the missing fact first."
                answer = None
        step = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {observation}\n"
        prompt += step
        if show:
            print(step)
        if answer is not None:
            return answer
    n = max_steps + 1
    forced = clean_forced(ask(prompt + f"Thought {n}: I have no steps left, so I must answer now with my best guess.\nAction {n}: Finish[",
                              stop=("]", "\n")))
    if show:
        print(f"Thought {n}: I have no steps left, so I must answer now with my best guess.\nAction {n}: Finish[{forced}]\n")
    return forced

print("Agent v4 ready: react_v4(...)")

Agent v4 ready: react_v4(...)


In [36]:
print("ANSWER:", react_v4(QUESTIONS[5][0], show=True))

Thought 1: I need to look this up.
Action 1: Invalid
Observation 1: Invalid action. Use Search[...], Lookup[...] or Finish[...].

Thought 2: I need to look this up.
Action 2: Search[Bronx scientific journal]
Observation 2: Could not find Bronx scientific journal. Similar: ['Bronx High School of Science', 'Bronx Zoo', 'List of Bronx High School of Science alumni', 'New York Botanical Garden', 'Pelham Bay Park']. (Automatically trying Bronx High School of Science instead.) The Bronx High School of Science is a public specialized high school founded in 1938 in the Bronx in New York City. Admission involves passing the Specialized High Schools Admissions Test. The school is operated by the New York City Department of Education. Founded by science educator Morris Meister who served as its first principal, Bronx Science is located in what is now Kingsbridge Heights, also known as Jerome Park, a neighborhood in the northwest portion of the Bronx. Known for its focus on mathematics and science